# lssem3d CuPy backend on an A100Validated already on an NVIDIA DGX Spark (GB10) in Docker — the whole validationladder passes there (`CUPY_BACKEND.md`). What the Spark **cannot** answer isperformance: its FP64 rate is ~1/41 of its FP32 (measured 0.21 against 8.61TFLOP/s) and its bandwidth measured 112 GB/s, *below* an M3 Max. This solver isfloat64 and bandwidth-bound, so a production number has to come fromFP64-capable hardware.**Runtime → Change runtime type → A100 GPU** before running. If you get an L4 orT4 instead, cell 1 says so — FP64 numbers from those are not worth recording.

In [ ]:
# 1. What did we actually get?  (Colab does not always give you the A100.)!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheaderimport subprocessname = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],                      capture_output=True, text=True).stdout.strip()print('\nGPU:', name)if 'A100' not in name:    print('!! NOT an A100 -- FP64 numbers from this device are not comparable.')    print('   Runtime -> Change runtime type -> A100, then Restart runtime.')

In [ ]:
# 2. FP64 sanity: the number that decides whether this hardware is usable.#    DGX Spark GB10 reference: FP64 0.21, FP32 8.61 TFLOP/s, 112 GB/s.#    An A100 should be ~9.7 TFLOP/s FP64 and 1.5-1.9 TB/s.import cupy as cp, timedef timeit(f, n=5):    f(); cp.cuda.Stream.null.synchronize()    t0 = time.perf_counter()    for _ in range(n): f()    cp.cuda.Stream.null.synchronize()    return (time.perf_counter()-t0)/nN = 4096a = cp.random.rand(N,N,dtype=cp.float64); b = cp.random.rand(N,N,dtype=cp.float64)print(f'FP64 GEMM  {2*N**3/timeit(lambda: a@b)/1e12:6.2f} TFLOP/s')a32, b32 = a.astype(cp.float32), b.astype(cp.float32)print(f'FP32 GEMM  {2*N**3/timeit(lambda: a32@b32)/1e12:6.2f} TFLOP/s')n = 1 << 26x = cp.random.rand(n,dtype=cp.float64); y = cp.random.rand(n,dtype=cp.float64); z = cp.empty_like(x)print(f'bandwidth  {3*n*8/timeit(lambda: cp.add(x,y,out=z))/1e9:6.0f} GB/s (fp64 triad)')

In [ ]:
# 3. Get the code.  Public repo; the CuPy work is on its own branch.!git clone -q --branch cupy-backend https://github.com/chandc/Python_SEM.git /content/lssem%cd /content/lssem!git log --oneline -1!pip install -q matplotlibimport cupy, numpy, scipyprint('cupy', cupy.__version__, '| numpy', numpy.__version__, '| scipy', scipy.__version__)

## Validation first, performance secondA backend is not trusted until it re-passes the ladder — symmetry and self-paritytests cannot find a consistently wrong operator. These gates are checked againstanalytic values, so they should give the *same numbers* here as on the Spark.

In [ ]:
# 4. Operator parity against the NumPy reference (fast).!python scratch/cupy_parity.py

In [ ]:
# 5. Gate 3 (balance) and Gate 1 (Stokes sigma vs the analytic 9.3137399).#    Spark results to match:#      sigma  9.3153041 / 9.3141300 / 9.3138373  at dt = 0.01/0.005/0.0025#      balance worst deviation 6.65e-06!python scratch/cupy_validation_ladder.py 3!python scratch/cupy_validation_ladder.py 1

In [ ]:
# 6. Gate 2 -- the expensive one (z-convection order at tol 1e-12).#    GB10 took 147 / 305 / 633 s per dt.  Expect much faster here.#    Expected: 5.724e-07 / 1.431e-07 / 3.577e-08, order 2.00.!python scratch/cupy_validation_ladder.py 2

In [ ]:
# 7. THE POINT OF THE EXERCISE: production-scale throughput.#    GB10 reference (same script, CuPy):#      0.53 M dof  2.36 ms | 1.23 M  3.85 ms | 3.43 M 11.84 ms | 6.17 M 22.39 ms#    6.17 M dof is the 88^3 production configuration.!python scratch/bench_backends.py numpy cupy

## Saving resultsColab VMs are ephemeral. Mount Drive before anything long, and write checkpointsand diagnostics there rather than to `/content`.

In [ ]:
# 8. Optional: persist output.from google.colab import drivedrive.mount('/content/drive')!mkdir -p /content/drive/MyDrive/lssem_results!cp -v /content/lssem/scratch/*.npz /content/drive/MyDrive/lssem_results/ 2>/dev/null || echo 'no npz yet'